In [0]:
# 1. Define las variables de configuración de Azure
service_principal_id = "TU_CLIENT_ID_DE_AZURE"
service_principal_key = "TU_CLIENT_SECRET_DE_AZURE"
tenant_id = "TU_TENANT_ID_DE_AZURE"
storage_account_name = "TU_CUENTA_DE_ALMACENAMIENTO_AZURE"
container_name = "TU_CONTAINER_AZURE"

# 2. Configura las propiedades de Spark para la conexión directa
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", service_principal_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", service_principal_key)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://microsoftonline.com{tenant_id}/oauth2/token")

In [0]:
# 1. Credenciales de Azure
client_id = "TU_CLIENT_ID_DE_AZURE"
client_secret = "TU_CLIENT_SECRET_DE_AZURE"
tenant_id = "TU_CONTAINER_AZURE"

# Variables extraídas de tu URL real
storage_account = "TU_CUENTA_DE_ALMACENAMIENTO_AZURE"
container = "raw"

# 2. Construcción automática de la ruta ABFSS oficial para Azure Data Lake
adls_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/ecdc/muertes.csv"

# 3. Lectura directa inyectando el mapa de opciones efímeras
df = spark.read \
    .option("fs.azure.account.auth.type", "OAuth") \
    .option("fs.azure.account.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") \
    .option("fs.azure.account.oauth2.client.id", client_id) \
    .option("fs.azure.account.oauth2.client.secret", client_secret) \
    .option("fs.azure.account.oauth2.client.endpoint", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token") \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(adls_path)

# 4. Mostrar los datos para validar la conexión de Lab 2
display(df)

In [0]:
# --- CELDA 2: GUARDAR DATA EN SQL DESDE LA MEMORIA ---

# Guardamos el dataframe 'df' cargado en memoria como una tabla administrada.
# Al guardarse localmente en el Metastore de Databricks, ya no necesitará consultar el ADLS en el futuro.
df.write.mode("overwrite").saveAsTable("tabla_covid_laboratorio")

# Confirmamos imprimiendo el esquema para estar seguros de qué tipo de datos detectó Spark
df.printSchema()

In [0]:
# --- CELDA 3: COMPROBACIÓN EN PYSPARK ---

# 1. Leer la tabla directamente desde el catálogo local (Metastore)
df_desde_tabla = spark.read.table("tabla_covid_laboratorio")

# 2. Imprimir el esquema guardado para demostrar que mantuvo los tipos de datos correctos
print("=== ESQUEMA DETECTADO EN PYSPARK ===")
df_desde_tabla.printSchema()

# 3. Mostrar un pequeño extracto de la data persistida
display(df_desde_tabla.limit(5))

In [0]:
%sql
-- --- CELDA 4: COMPROBACIÓN EN SQL ANSEL ---

-- 1. Describir la estructura de la tabla para validar las columnas y tipos de datos
DESCRIBE TABLE tabla_covid_laboratorio;

In [0]:
# --- CELDA 5: EJERCICIO 1 - FILTRADO Y SELECCIÓN BÁSICA ---
from pyspark.sql.functions import col

# 1. Aplicamos los filtros encadenados usando operadores lógicos (&)
# Nota: Como la columna 'date' es de tipo string o timestamp, buscamos que empiece con '2020'
df_ejercicio_1 = df.filter(
    (col("indicator") == "deaths") & 
    (col("date").like("2020%")) & 
    (col("country") == "Colombia")
).select(
    "country", 
    "continent", 
    "date", 
    "daily_count"
)

# 2. Mostramos el resultado en pantalla para validar nuestro pipeline
display(df_ejercicio_1)

In [0]:
# --- CELDA 6: EJERCICIO 2 - AGREGACIONES Y ESTADÍSTICAS ---
from pyspark.sql.functions import col, sum, when

# 1. Agrupamos por continente y sumamos condicionalmente según el tipo de indicador
df_continentes = df.groupBy("continent").agg(
    sum(when(col("indicator") == "confirmed cases", col("daily_count")).otherwise(0)).alias("total_casos_confirmados"),
    sum(when(col("indicator") == "deaths", col("daily_count")).otherwise(0)).alias("total_muertes")
).orderBy(col("total_muertes").desc()) # Ordenamos de mayor a menor impacto de fallecimientos

# 2. Mostramos el resultado consolidado por continente
display(df_continentes)

In [0]:
# --- CELDA 7: EJERCICIO 3 - COLUMNAS DERIVADAS Y LIMPIEZA ---
from pyspark.sql.functions import col, round

# 1. Usamos .withColumn() para añadir la nueva columna basada en una operación aritmética
# Además, aplicamos round() para limitar el resultado a 4 decimales
df_enriquecido = df.withColumn(
    "porcentaje_poblacion_afectada",
    round((col("daily_count") / col("population")) * 100, 4)
)

# 2. Seleccionamos las columnas clave para verificar visualmente el cálculo
df_resultado_3 = df_enriquecido.select(
    "country", 
    "date", 
    "indicator", 
    "daily_count", 
    "population", 
    "porcentaje_poblacion_afectada"
).filter(col("daily_count") > 0) # Filtramos solo los días con movimientos para ver el cálculo real

# 3. Mostrar el resultado en el pipeline
display(df_resultado_3.limit(10))

In [0]:
# --- CELDA 7 MODIFICADA: REFINAMIENTO EJERCICIO 3 ---
from pyspark.sql.functions import col, round

# 1. Creamos la columna derivada con el porcentaje redondeado a 4 decimales
df_enriquecido = df.withColumn(
    "porcentaje_poblacion_afectada",
    round((col("daily_count") / col("population")) * 100, 4)
)

# 2. Aplicamos el filtro para "America" y ordenamos descendentemente por la nueva métrica
df_resultado_america = df_enriquecido.filter(
    (col("continent") == "America") & 
    (col("daily_count") > 0)
).select(
    "country", 
    "date", 
    "indicator", 
    "daily_count", 
    "population", 
    "porcentaje_poblacion_afectada"
).orderBy(col("porcentaje_poblacion_afectada").desc())

# 3. Mostrar el top de los registros con mayor impacto relativo en América
display(df_resultado_america)

In [0]:
# --- CELDA 8: EJERCICIO 4 - SIMULACIÓN DE AUTO-JOIN ---
from pyspark.sql.functions import col

# 1. Creamos el DataFrame de Casos Confirmados y renombramos su conteo diario
df_casos = df.filter(col("indicator") == "confirmed cases") \
             .select("country", "date", "daily_count") \
             .withColumnRenamed("daily_count", "casos_confirmados_diarios")

# 2. Creamos el DataFrame de Muertes y renombramos su conteo diario
df_muertes = df.filter(col("indicator") == "deaths") \
               .select("country", "date", "daily_count") \
               .withColumnRenamed("daily_count", "muertes_diarias")

# 3. Realizamos el INNER JOIN usando 'country' y 'date' como la llave compuesta (composite key)
df_joint_final = df_casos.join(
    df_muertes, 
    on=["country", "date"], 
    how="inner"
)

# 4. Ordenamos el resultado final por país y fecha para que se vea estructurado
df_resultado_pipeline = df_joint_final.orderBy("country", "date")

# 5. Mostramos el resultado del Join en el pipeline
display(df_resultado_pipeline)

In [0]:
# --- CELDA 9: TIME INTELLIGENCE Y CARGA FINAL AL ADLS ---
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

# 1. Definimos la "Ventana": Agrupamos por país y ordenamos cronológicamente por fecha
ventana_pais = Window.partitionBy("country").orderBy("date")

# 2. Aplicamos la función LAG para traer el valor de 'muertes_diarias' del registro anterior (1 día atrás)
df_final_analytics = df_resultado_pipeline.withColumn(
    "muertes_dia_anterior", 
    lag("muertes_diarias", 1).over(ventana_pais)
)

# 3. Definimos la ruta de destino en el contenedor de Azure Data Lake (ADLS)
# Vamos a guardarlo en una nueva carpeta llamada 'gold' o 'processed' dentro de tu container 'raw'
ruta_destino_adls = f"abfss://{container}@{storage_account}.dfs.core.windows.net/processed/covid_analytics"

# 4. ESCRITURA EN EL ADLS: Guardamos en formato PARQUET (eficiente, comprimido y rápido)
# Reinyectamos las credenciales efímeras para que Spark tenga permisos de ESCRITURA en tu Azure
df_final_analytics.write \
    .option("fs.azure.account.auth.type", "OAuth") \
    .option("fs.azure.account.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") \
    .option("fs.azure.account.oauth2.client.id", client_id) \
    .option("fs.azure.account.oauth2.client.secret", client_secret) \
    .option("fs.azure.account.oauth2.client.endpoint", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token") \
    .mode("overwrite") \
    .format("parquet") \
    .save(ruta_destino_adls)

print("¡ÉXITO TOTAL! El pipeline ha finalizado. Data procesada y guardada en ADLS.")

# 5. Mostramos una muestra del resultado final con el Time Intelligence aplicado
display(df_final_analytics.filter(col("muertes_diarias") > 0).limit(10))
